# GLM-5.3-Flash 企业部署：8×H20 / FP8 / vLLM

按顺序执行，每个代码单元前说明用途、输出和使用 GPU 的时点。操作目标为**专用 Ubuntu 24.04 x86_64 服务器**，不是共享训练服务器。模型为 `zai-org/GLM-5.3-Flash`；API 名为 `glm-5.3-flash`。

- `/srv/phyxi` 必须是管理员预先挂载的数据卷；本 Notebook 不格式化磁盘。
- 推荐主机内存 ≥768 GiB（脚本门槛 600 GiB）、数据卷 ≥1.5 TB，首次制备至少 600 GiB 空闲。单份 FP8 权重约 306 GiB，实际文件量在下载后记录。
- 安装驱动可能要求重启，企业 CA 签发需要管理员提交 CSR。这两步有明确停点，完成后继续，不要忽略错误。
- 完成准备后，只有“启动服务”单元会初始化 GPU 并加载模型；本地静态验证不代表目标服务器已验收。
- 初始上下文 64K、调度序列上限 16 是可调整部署参数，**不是实测承载人数**。
- 图片请求使用 Python 前端；H20 使用 FP8 权重与 BF16 KV，非 FP8 KV。不开启 MTP/PD 分离，先完成基础服务验收。

运行前先按 README 安装并启动 Jupyter；此 Notebook 的管理员 kernel 仅监听回环，通过 SSH 隧道访问。

## 0. 定位仓库和运行环境

本单元不修改系统。将内核目录定位到仓库，后续动作由 `scripts/deploy.py` 的具名阶段执行；所有命令在脚本中可审阅。遇到任何命令失败即停止，不继续后续阶段。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
REPO = Path.cwd().resolve()
if REPO.name == "notebooks": REPO = REPO.parent
assert (REPO / "scripts/deploy.py").is_file(), "请在本仓库内打开 Notebook"
assert os.geteuid() == 0, "需要按 README 启动管理员 Notebook 内核"
CONFIG_PATH = REPO / "config.local.json"  # 已加入 .gitignore；不要提交执行后的 Notebook

def stage(name):
    subprocess.run(["/usr/bin/python3", str(REPO / "scripts/deploy.py"), name,
                    "--config", str(CONFIG_PATH)], check=True)
print("Repository:", REPO)
print((REPO / "deployment/versions.json").read_text())

## 1. 填写企业部署输入

只需在这里集中填写域名、内网 IP、允许网段和 8 张 GPU UUID。域名必须由企业 DNS 解析到内网 IP；防火墙/安全组允许业务网段到 TCP 443，以及管理网到 SSH 22。Jupyter 8888、模型 8000 不开放公网。

`tls_mode="enterprise"` 为默认交付方式；封闭验收环境可显式改成 `self-signed`（30 天测试证书），正式交付前换成企业 CA。

首次没有驱动时先保留 GPU 列表为空；安装驱动并重启后获取 UUID，再重跑本单元填写。不要按物理编号猜 UUID。

In [ ]:
if CONFIG_PATH.exists():
    config = json.loads(CONFIG_PATH.read_text())
else:
    config = json.loads((REPO / "deployment/config.example.json").read_text())

# 首次运行：改为企业实际地址。恢复 Notebook 时保留已保存值。
# config["domain"] = "llm.your-company.example"
# config["intranet_ip"] = "10.20.0.10"
# config["allowed_cidrs"] = ["10.20.0.0/16"]
# config["gpu_uuids"] = ["GPU-...", "GPU-...", "GPU-...", "GPU-...",
#                         "GPU-...", "GPU-...", "GPU-...", "GPU-..."]
# config["alertmanager"] = "alertmanager.corp.example:9093"  # 已有企业告警接收器

CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
CONFIG_PATH.chmod(0o600)
print("配置已保存；仅显示非敏感状态：", {k: config[k] for k in ["domain", "intranet_ip", "max_model_len", "max_num_seqs", "tls_mode"]})

## 2. 服务器与数据卷检查

检查 Ubuntu 24.04、x86_64、主机内存、独立挂载点与可用空间；拒绝已存在其他容器或 GPU 工作负载的服务器。输出硬件信息。未分配数据卷时先由管理员完成挂载，再重跑本步。

In [ ]:
stage("preflight")

## 3. 系统基础依赖与目录

安装证书、网络、Python、内核头文件等组件；创建 `/srv/phyxi/{docker,containerd,models,cache,prometheus,secrets,artifacts,evidence,releases}`，启用 SSH 和时间同步。

In [ ]:
stage("base")

## 4. NVIDIA 驱动

复用已正常工作的 580 或更新驱动；若缺失，安装官方 580 分支驱动。本步安装后主动抛出“需要重启”提示：在管理员终端执行 `reboot`，重启 Jupyter，重新运行 0、1、4 单元。Secure Boot 按提示完成 MOK 注册。不要在 Notebook 中自动重启。

In [ ]:
stage("driver")

## 5. 记录 GPU UUID

读取 GPU 清单，不加载模型。将下方输出的八张 H20 UUID 填入第 1 节 `gpu_uuids`，重新执行第 1 节保存。企业方案使用一份模型跨八卡 TP=8。

In [ ]:
subprocess.run(["nvidia-smi", "--query-gpu=index,uuid,name,memory.total,memory.used", "--format=csv"], check=True)
assert len(json.loads(CONFIG_PATH.read_text())["gpu_uuids"]) == 8, "先更新第 1 节，填入八张卡的 UUID"

## 6. 安装 Docker 与持久存储

从 Docker 官方仓库选择 29.x，或使用第 1 节指定的批准版本。首次配置镜像与容器存储到 `/srv/phyxi/docker`，containerd 到 `/srv/phyxi/containerd`。记录版本；重跑核对已管理的 Engine，不搬迁已有业务数据。此阶段会重启专用服务器的 Docker。

In [ ]:
stage("docker")

## 7. 安装 GPU 容器运行时

安装固定版本 NVIDIA Container Toolkit，注册 NVIDIA runtime，记录版本并锁定软件包。重启专用服务器 Docker；不加载模型。

In [ ]:
stage("toolkit")

## 8. 固定镜像与 GLM 兼容性检查

拉取 `deployment/versions.json` 中固定 digest 的 vLLM 镜像；Nginx/Prometheus 首次解析 digest 后保存，后续复用。运行容器检查 GLM5Next 架构、glm47 工具/思考解析器、FlashInfer 最低版本和启动参数；不分配 GPU。结果在 `artifacts/image-compatibility.json`。失败时停止，不能仅凭镜像标签继续。

In [ ]:
stage("images")

## 9. 下载固定版本 FP8 权重

下载官方仓库指定 commit；已有同一版本时断点续传。核对 FP8 配置和全部权重分片，逐文件生成 SHA-256。模型目录、总字节数、分片数写入 `artifacts/model.json`。此步磁盘与网络开销较大，不占 GPU。

In [ ]:
stage("model")

## 10. 生成正式配置与两层密钥

根据第 1 节生成 Compose 的八卡 UUID 列表、模型参数、Nginx 域名与允许网段。建立 `current` 发布软链接；生成后端 API Key、工作站 Token 及网关映射，不打印秘密。配置位于 `releases/<release>/`，凭据位于 `secrets/`；重复运行保留原有凭据。

In [ ]:
stage("render")

## 11. 生成 TLS 私钥与 CSR

生成服务器私钥和证书签发请求。企业 CA 模式：将 `/srv/phyxi/artifacts/server.csr` 提交 CA；收到完整证书链和根/中间 CA 信任文件后，放到第 1 节配置的 `/secure-transfer/` 路径。私钥始终留在服务器。

In [ ]:
stage("tls_csr")

## 12. 安装并验证 TLS 证书

校验证书域名、链和私钥配对；设置 Nginx 读取权限。企业 CA 文件未就绪则停止；自签名验收模式会生成 30 天证书，并把该证书用作工作站信任锚。

In [ ]:
stage("tls_install")

## 13. 检查将要执行的配置

本单元只查看配置。关键基线：TP=8，FP8 权重自动按 checkpoint 读取；`dtype=bfloat16` 为未量化计算与 KV 默认精度；`VLLM_USE_RUST_FRONTEND=0`；两种解析器均为 glm47。不是 Qwen 参数。

`MAX_NUM_SEQS=16` 为模型调度初值；Nginx 单工作站在途 4、总在途 16。多个工作站须分别发放 Token。参数解释见 `docs/parameters.md`。

In [ ]:
print(Path("/srv/phyxi/current/start-model.sh").read_text())
print(Path("/srv/phyxi/current/.env").read_text())  # .env 仅版本、地址和参数，不保存密钥
subprocess.run(["bash", "/srv/phyxi/current/dc.sh", "config", "--quiet"], check=True)

## 14. 启动服务（本步开始使用八张 GPU）

先核对八卡空闲和 DNS，再运行轻量 CUDA 检查、Nginx 配置检查，安装并启动 `phyxi-model.service`。启动最长等待约 65 分钟，用于首次权重加载和内核制备；网关在模型 healthy 后启动。

失败时在管理员终端查看 `bash /srv/phyxi/current/dc.sh logs --tail 100 model`。不要反复绕过检查，先修复日志中的具体错误。初次安装保持批准的软件源网络可达，完成启动验收后再执行企业出口策略。

In [ ]:
stage("start")

## 15. TLS、鉴权、图片及多轮工具协议验收

检查错误凭据返回 401、合法模型列表、文本、SSE、红色色块图片、工具调用与第二轮工具回填。使用 GLM 的 `reasoning_effort=low`，最多 4,096 输出 token（含思考）；不使用 Qwen 的 enable_thinking 开关。保存 `evidence/api-validation.json`。检查 Prometheus 就绪。

In [ ]:
stage("verify")

## 16. 有界并发验收

按客户端并发 1、2、4，分别预热一次并执行 16 个单轮请求，最多 1,024 输出 token。完成后保存每请求状态、实际 token、耗时、截断数与可见回答数。TPS 是服务合计 completion_tokens / 测试窗口，包含思考 token、网络与排队；不是单用户文字速度。结果为本机实测，不引用历史 27B 数据。

此组为协议/短负载验收；长期吞吐与多轮建模用户容量需另用项目实际任务集测试。

In [ ]:
stage("benchmark")

## 17. 导出工作站配置

生成仅管理员可读的 `/srv/phyxi/client-export/`，包含 CA、该工作站 Token、启动脚本和网关地址。不包含后端 Key、TLS 私钥。由企业安全渠道分发给指定工作站；新增工作站需发放不同 Token。

Windows 安装和项目接入执行仓库 `scripts/setup-workstation.ps1`，详见 `docs/workstation.md`。服务端 Notebook 不能代替在 Windows 上安装客户端和 CATIA。项目对话链路已提供版本校验的连接适配；实际 CATIA 工具链是独立业务验收。

In [ ]:
stage("export")

## 18. 状态与交付记录

确认模型、网关、监控状态，保存软件版本与 GPU 采样。根据 `docs/operations.md` 完成告警送达、证书续签、主机重启恢复和回滚演练，再签署企业交付验收。

In [ ]:
stage("status")
subprocess.run(["bash", "-c", "dpkg-query -W -f='${Package}\t${Version}\n' docker-ce docker-ce-cli containerd.io docker-compose-plugin nvidia-container-toolkit > /srv/phyxi/artifacts/system-packages.tsv"], check=True)
print("配置、版本与验收结果分别在 /srv/phyxi/current、artifacts、evidence")

## 19. 可选：停止服务并释放 GPU

默认不停止正式服务。需要结束验收时将下面 `STOP_SERVICE` 改为 True。停止 systemd 管理的模型、网关和监控，保留权重、配置、缓存和证据。重启正式服务使用 `systemctl start phyxi-model.service`；启动前检查分配的 GPU 空闲。

In [ ]:
STOP_SERVICE = False
if STOP_SERVICE:
    stage("stop")
else:
    print("未执行停止操作")